# Live Demo — WRDS Data Pull
### Session 5 · ExInt II · WU Vienna · SS 2026

**What we do in this notebook:**
1. Connect to WRDS
2. Explore what Compustat Global looks like
3. Pull 5 firms for fiscal years 2015–2024 — all variables
4. Inspect the data
5. Save as parquet in a timestamped folder

This is the **mini version** of `01_pull_data.py` — same logic, small data so we can see results instantly.

> **Reference project:** https://github.com/vkiefner/sme-intl

---
## Cell 1 — Imports & credentials

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv
import wrds
import pandas as pd
from datetime import datetime

def find_env():
    current = Path(os.getcwd())
    # Search upward AND in sibling folders at each level
    for path in [current] + list(current.parents):
        # Check current level
        if (path / ".env").exists():
            return path / ".env"
        # Check all siblings at this level
        for sibling in path.iterdir():
            if sibling.is_dir() and (sibling / ".env").exists():
                return sibling / ".env"
    raise FileNotFoundError("Could not find .env anywhere.")

env_file = find_env()
project_root = env_file.parent
os.chdir(project_root)

print(f"Found .env at:     {env_file}")
print(f"Working directory: {os.getcwd()}")

load_dotenv(dotenv_path=env_file, override=True)
WRDS_USER = os.getenv("WRDS_USERNAME")
print(f"WRDS user:         {WRDS_USER}")

Found .env at:     /Users/larawild/sme-internationalization/.env
Working directory: /Users/larawild/sme-internationalization
WRDS user:         larawild


---
## Cell 2 — Connect to WRDS

In [2]:
db = wrds.Connection(wrds_username=WRDS_USER)
print("Connected!")

Loading library list...
Done
Connected!


---
## Cell 3 — What tables are available in Compustat Global?

In [3]:
# List all tables in the Compustat Global library
tables = db.list_tables(library="comp_global_daily")
print(f"{len(tables)} tables available:")
for t in tables:
    print(" ", t)

125 tables available:
  dd_group
  dd_group_xref
  dd_item
  dd_package
  g_chars
  g_co_aaudit
  g_co_adesind
  g_co_afnd1
  g_co_afnd2
  g_co_afnddc1
  g_co_afnddc2
  g_co_afntind1
  g_co_afntind2
  g_co_ainvval
  g_co_gsuppl
  g_co_hgic
  g_co_iaudit
  g_co_idesind
  g_co_ifndq
  g_co_ifndsa
  g_co_ifndytd
  g_co_ifntq
  g_co_ifntsa
  g_co_ifntytd
  g_co_industry
  g_co_ipcd
  g_co_offtitl
  g_company
  g_currency
  g_ecind_desc
  g_ecind_mth
  g_exrt_dly
  g_exrt_mth
  g_funda
  g_funda_fncd
  g_fundq
  g_fundq_fncd
  g_idx_daily
  g_idx_index
  g_idx_mth
  g_idxcst_his
  g_indexcst_his
  g_names
  g_names_ix
  g_names_ix_cst
  g_namesq
  g_sec_adesind
  g_sec_adjfact
  g_sec_afnd
  g_sec_afnddc
  g_sec_afnt
  g_sec_divid
  g_sec_dprc
  g_sec_dtrt
  g_sec_gmdivfn
  g_sec_gmth
  g_sec_gmthdiv
  g_sec_gmthprc
  g_sec_history
  g_sec_idesind
  g_sec_ifnd
  g_sec_ifnt
  g_sec_split
  g_secd
  g_secm
  g_secnamesd
  g_security
  g_sedolgvkey
  g_tmptable_pkg6775_tbl5551
  gsecnamesm
  r

---
## Cell 4 — What columns does g_funda have?

In [4]:
# g_funda = Global Fundamentals Annual — the main firm-year table
schema = db.describe_table(library="comp_global_daily", table="g_funda")
print(f"{len(schema)} columns available in g_funda")
schema.head(20)

Approximately 1081405 rows in comp_global_daily.g_funda.
444 columns available in g_funda


,name,nullable,type,comment
0,gvkey,True,VARCHAR(7),Global Company Key
1,indfmt,True,VARCHAR(13),Industry Format
2,datafmt,True,VARCHAR(13),Data Format
3,consol,True,VARCHAR(3),Level of Consolidation - Company Annual Descri...
4,popsrc,True,VARCHAR(2),Population Source
5,acctstd,True,VARCHAR(9),Accounting Standard
6,acqmeth,True,VARCHAR(3),Acquisition Method
7,bspr,True,VARCHAR(9),Balance Sheet Presentation
8,compst,True,VARCHAR(9),Comparability Status
9,curcd,True,VARCHAR(4),ISO Currency Code


---
## Cell 5 — Choose 5 firms to pull

We use `gvkey` — the Compustat firm identifier.  
These are 5 well-known European firms spanning different countries and sectors.

| gvkey  | Firm     | Country |
|--------|----------|---------|
| 100737 | Volkswagen  | DEU     |
| 016603 | Nestlé   | CHE     |
| 024625 | TotalEnergies   | FRA     |
| 014447 | LVMH  | FRA     |
| 017436 | BASF | DEU     |

In [5]:
# Real verified gvkeys from Compustat Global
# Volkswagen (DEU), Nestlé (CHE), TotalEnergies (FRA), LVMH (FRA), BASF (DEU)
FIVE_FIRMS = ('100737', '016603', '024625', '014447', '017436')

firms_sql = ", ".join(f"'{g}'" for g in FIVE_FIRMS)
print(f"Pulling: Volkswagen, Nestlé, TotalEnergies, LVMH, BASF")
print(f"gvkeys:  {firms_sql}")

Pulling: Volkswagen, Nestlé, TotalEnergies, LVMH, BASF
gvkeys:  '100737', '016603', '024625', '014447', '017436'


---
## Cell 6 — Pull ALL variables for these 5 firms, 2015–2024

In [6]:
# Step 1: try without any filters first to confirm the firms exist
query = f"""
    SELECT *
    FROM comp_global_daily.g_funda
    WHERE gvkey   IN ({firms_sql})
      AND fyear   BETWEEN 2015 AND 2024
      AND datafmt = 'HIST_STD'
      AND indfmt  = 'INDL'
      AND popsrc  = 'I'
      AND consol  = 'C'
    ORDER BY gvkey, fyear
"""

print("Pulling from WRDS...")
df_raw = db.raw_sql(query, date_cols=["datadate"])
print(f"\nDone! Shape: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")

Pulling from WRDS...

Done! Shape: 50 rows × 444 columns


---
## Cell 7 — First look at the data

In [7]:
# First 5 rows
df_raw.head()

,gvkey,indfmt,datafmt,consol,popsrc,acctstd,acqmeth,bspr,compst,curcd,...,conm,costat,fic,loc,naicsh,sich,rank,au,auop,hiid
0,014447,INDL,HIST_STD,C,I,DI,<NA>,GO,<NA>,EUR,...,LVMH MOET HENNESSY LOUIS V,A,FRA,FRA,3152,2330,1,5,4,02W
1,014447,INDL,HIST_STD,C,I,DI,<NA>,GO,<NA>,EUR,...,LVMH MOET HENNESSY LOUIS V,A,FRA,FRA,3152,2330,1,9,4,02W
2,014447,INDL,HIST_STD,C,I,DI,<NA>,GO,<NA>,EUR,...,LVMH MOET HENNESSY LOUIS V,A,FRA,FRA,3152,2330,1,9,4,02W
3,014447,INDL,HIST_STD,C,I,DI,<NA>,GO,<NA>,EUR,...,LVMH MOET HENNESSY LOUIS V,A,FRA,FRA,3152,2330,1,9,4,02W
4,014447,INDL,HIST_STD,C,I,DI,<NA>,GO,<NA>,EUR,...,LVMH MOET HENNESSY LOUIS V,A,FRA,FRA,3152,2330,1,9,4,02W


In [8]:
# Column names — all available variables
print(f"Total columns: {len(df_raw.columns)}")
print("\nAll column names:")
for i, col in enumerate(df_raw.columns):
    print(f"  {i:>3}. {col}")

Total columns: 444

All column names:
    0. gvkey
    1. indfmt
    2. datafmt
    3. consol
    4. popsrc
    5. acctstd
    6. acqmeth
    7. bspr
    8. compst
    9. curcd
   10. final
   11. fyear
   12. fyr
   13. ismod
   14. pddur
   15. scf
   16. src
   17. stalt
   18. upd
   19. datadate
   20. fdate
   21. pdate
   22. accli
   23. acco
   24. aco
   25. acofs
   26. acox
   27. acoxfs
   28. acqdisn
   29. acqdiso
   30. act
   31. adpac
   32. am
   33. amdc
   34. ao
   35. aoloch
   36. aox
   37. ap
   38. apalch
   39. apch
   40. apdpfs
   41. apfs
   42. apo
   43. apofs
   44. aqc
   45. artfs
   46. asdis
   47. asinv
   48. at
   49. atoch
   50. autxr
   51. bcef
   52. bct
   53. ca
   54. capcst
   55. capfl
   56. capr1
   57. capr2
   58. capr3
   59. caprt
   60. caps
   61. capx
   62. capxfi
   63. ceq
   64. cfbd
   65. cfere
   66. cflaoth
   67. cfo
   68. cfpdo
   69. cga
   70. ch
   71. che
   72. cheb
   73. chech
   74. chee
   75. chefs
   76. 

In [9]:
# Data types
df_raw.dtypes

gvkey      string[python]
indfmt     string[python]
datafmt    string[python]
consol     string[python]
popsrc     string[python]
                ...      
sich                Int64
rank                Int64
au         string[python]
auop       string[python]
hiid       string[python]
Length: 444, dtype: object

In [10]:
# Which firms did we get? How many years each?
df_raw.groupby(["gvkey", "conm"])["fyear"].agg(["min", "max", "count"]).rename(
    columns={"min": "first_year", "max": "last_year", "count": "n_years"}
)

,,first_year,last_year,n_years
gvkey,conm,,,
014447,LVMH MOET HENNESSY LOUIS V,2015,2024,10
016603,NESTLE SA/AG,2015,2024,10
017436,BASF SE,2015,2024,10
024625,TOTALENERGIES SE,2015,2024,10
100737,VOLKSWAGEN AG,2015,2024,10


---
## Cell 8 — Look at key financial variables

In [11]:
# Select the most commonly used variables for a quick preview
key_vars = ["gvkey", "conm", "fyear", "loc",
            "at",    # total assets
            "sale",  # net sales
            "ib",    # income before extraordinary items (net income)
            "xrd",   # R&D expenditure
            "emp",   # employees (thousands)
            "dltt",  # long-term debt
            "pifo",  # pre-tax income from foreign operations
            "curcd" # currency
           ]

# Keep only columns that exist in our pull
available = [v for v in key_vars if v in df_raw.columns]
df_raw[available]

,gvkey,conm,fyear,loc,at,sale,ib,xrd,emp,dltt,curcd
0,014447,LVMH MOET HENNESSY LOUIS V,2015,FRA,57601.0,35664.0,3573.0,97.0,100.983,4511.0,EUR
1,014447,LVMH MOET HENNESSY LOUIS V,2016,FRA,59622.0,37600.0,3981.0,111.0,107.053,3932.0,EUR
2,014447,LVMH MOET HENNESSY LOUIS V,2017,FRA,68550.0,42636.0,5129.0,130.0,145.247,7046.0,EUR
3,014447,LVMH MOET HENNESSY LOUIS V,2018,FRA,74300.0,46826.0,6354.0,130.0,127.739,6005.0,EUR
4,014447,LVMH MOET HENNESSY LOUIS V,2019,FRA,96507.0,53670.0,7171.0,140.0,163.309,15474.0,EUR
5,014447,LVMH MOET HENNESSY LOUIS V,2020,FRA,108671.0,44651.0,4702.0,139.0,148.343,24730.0,EUR
6,014447,LVMH MOET HENNESSY LOUIS V,2021,FRA,125311.0,64215.0,12036.0,147.0,157.953,24052.0,EUR
7,014447,LVMH MOET HENNESSY LOUIS V,2022,FRA,134646.0,79184.0,14084.0,172.0,173.492,23156.0,EUR
8,014447,LVMH MOET HENNESSY LOUIS V,2023,FRA,143694.0,86153.0,15174.0,<NA>,192.287,25037.0,EUR
9,014447,LVMH MOET HENNESSY LOUIS V,2024,FRA,149190.0,84683.0,12550.0,<NA>,200.518,26951.0,EUR


In [12]:
# Quick summary stats on numeric columns
df_raw[["at", "sale", "ib", "xrd", "emp"]].describe().round(2)

,at,sale,ib,xrd,emp
count,50.0,50.0,50.0,48.0,50.0
mean,215752.36,129264.46,8732.56,3766.79,262.42
std,162111.23,81028.98,5964.94,5083.57,212.11
min,57601.0,35664.0,-7242.0,97.0,95.39
25%,87058.25,66170.5,4798.25,877.25,107.88
50%,136098.5,90572.5,8966.5,1682.5,153.15
75%,285028.75,192866.0,12470.5,2172.5,319.25
max,632905.0,324656.0,21384.0,17963.0,684.02


---
## Cell 9 — How many missing values per column?

In [13]:
# Missing value count and percentage
missing = pd.DataFrame({
    "missing_n":   df_raw.isnull().sum(),
    "missing_pct": (df_raw.isnull().sum() / len(df_raw) * 100).round(1)
})

# Show only columns that have at least one missing value
missing[missing["missing_n"] > 0].sort_values("missing_pct", ascending=False)

,missing_n,missing_pct
pv,50,100.0
acqmeth,50,100.0
xobd,50,100.0
xnitb,50,100.0
xivre,50,100.0
...,...,...
apch,8,16.0
xrent,7,14.0
am,4,8.0
xrd,2,4.0


---
## Cell 10 — Standardize column names

In [14]:
# Lowercase + strip whitespace — same as the full pull script
df = df_raw.copy()
df.columns = [c.strip().lower() for c in df.columns]

print("Column names standardized.")
print("Before:", list(df_raw.columns[:5]))
print("After: ", list(df.columns[:5]))

Column names standardized.
Before: ['gvkey', 'indfmt', 'datafmt', 'consol', 'popsrc']
After:  ['gvkey', 'indfmt', 'datafmt', 'consol', 'popsrc']


---
## Cell 11 — Save to timestamped parquet folder

Same structure as `01_pull_data.py`: each run gets its own folder named with the current date and time.

In [15]:
# Create timestamped output folder
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
out_dir   = Path("data") / "raw" / timestamp
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Output folder: {out_dir}")

# Save one parquet file per year (same chunking strategy as full pull)
for year, group in df.groupby("fyear"):
    out_path = out_dir / f"fyear_{int(year)}.parquet"
    group.to_parquet(out_path, index=False)
    print(f"  fyear_{int(year)}.parquet  →  {len(group)} rows")

# Write metadata
(out_dir / "pull_metadata.txt").write_text(
    f"LIVE DEMO PULL (5 firms)\n"
    f"========================\n"
    f"Pulled:       {datetime.now().isoformat()}\n"
    f"Source:       comp_global_daily.g_funda\n"
    f"Firms:        {', '.join(FIVE_FIRMS)}\n"
    f"Fiscal years: 2015–2024\n"
    f"Rows:         {len(df)}\n"
    f"Columns:      {df.shape[1]}\n"
)

print(f"\nDone! {len(df)} rows saved across {df['fyear'].nunique()} year files.")

Output folder: data/raw/2026-06-02_10-10-49
  fyear_2015.parquet  →  5 rows
  fyear_2016.parquet  →  5 rows
  fyear_2017.parquet  →  5 rows
  fyear_2018.parquet  →  5 rows
  fyear_2019.parquet  →  5 rows
  fyear_2020.parquet  →  5 rows
  fyear_2021.parquet  →  5 rows
  fyear_2022.parquet  →  5 rows
  fyear_2023.parquet  →  5 rows
  fyear_2024.parquet  →  5 rows

Done! 50 rows saved across 10 year files.


In [16]:
!python -m pip install pyarrow


---
## Cell 12 — Verify: read back one parquet file

In [17]:
# Read back the 2024 file to confirm it saved correctly
check = pd.read_parquet(out_dir / "fyear_2024.parquet")
print(f"fyear_2024.parquet: {check.shape[0]} rows × {check.shape[1]} columns")
check[["gvkey", "conm", "fyear", "at", "sale", "ib"]].head()

fyear_2024.parquet: 5 rows × 444 columns


,gvkey,conm,fyear,at,sale,ib
0,014447,LVMH MOET HENNESSY LOUIS V,2024,149190.0,84683.0,12550.0
1,016603,NESTLE SA/AG,2024,139264.0,91354.0,10884.0
2,017436,BASF SE,2024,80415.0,65260.0,1298.0
3,024625,TOTALENERGIES SE,2024,285487.0,195610.0,15758.0
4,100737,VOLKSWAGEN AG,2024,632905.0,324656.0,11351.0


---
## Cell 13 — Close the connection

In [18]:
db.close()
print("WRDS connection closed.")
print(f"\nYour data is in: {out_dir}")
print(f"Next step:       run 02_clean.py (or the full 01_pull_data.py for your own project)")

WRDS connection closed.

Your data is in: data/raw/2026-06-02_10-10-49
Next step:       run 02_clean.py (or the full 01_pull_data.py for your own project)


---
## Summary — What just happened

| Step | What we did |
|------|-------------|
| Connected | `wrds.Connection()` using credentials from `.env` |
| Explored | Listed tables and columns available in Compustat Global |
| Queried | `SELECT *` for 5 firms, 2015–2024, with standard filters |
| Inspected | Shape, column names, dtypes, missing values |
| Cleaned names | Lowercase column names |
| Saved | Parquet files in a timestamped folder under `data/raw/` |

**Your full script `01_pull_data.py` does exactly the same thing** — just for ALL firms globally instead of 5, and chunked by fiscal year to handle the larger volume.

---
*ExInt II · WU Vienna · SS 2026 · github.com/vkiefner/sme-intl*